In [14]:
from pathlib import Path
import nibabel as nib
import numpy as np

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
)

PATIENT_ID = "BraTS-GLI-01250-000"

PATIENT_DIR = RAW_DIR / PATIENT_ID

MODALITIES = [
    "t1n",
    "t1c",
    "t2w",
    "t2f"
]

print("Patient directory:")
print(PATIENT_DIR)

print()
print("Files:")

for modality in MODALITIES:
    path = (
        PATIENT_DIR
        / f"{PATIENT_ID}-{modality}.nii.gz"
    )

    print(
        modality,
        "→",
        path.exists()
    )

seg_path = (
    PATIENT_DIR
    / f"{PATIENT_ID}-seg.nii.gz"
)

print(
    "seg →",
    seg_path.exists()
)

Patient directory:
/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/raw/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/BraTS-GLI-01250-000

Files:
t1n → True
t1c → True
t2w → True
t2f → True
seg → True


In [15]:
volumes = {}

for modality in MODALITIES:

    path = (
        PATIENT_DIR
        / f"{PATIENT_ID}-{modality}.nii.gz"
    )

    volume = nib.load(
        path
    ).get_fdata(
        dtype=np.float32
    )

    volumes[modality] = volume

seg = nib.load(
    seg_path
).get_fdata(
    dtype=np.float32
)

print("=" * 60)
print("V4 3D VOLUME INSPECTION")
print("=" * 60)

for modality, volume in volumes.items():

    print(
        f"{modality}:",
        volume.shape,
        volume.dtype
    )

print(
    "seg:",
    seg.shape,
    seg.dtype
)

V4 3D VOLUME INSPECTION
t1n: (240, 240, 155) float32
t1c: (240, 240, 155) float32
t2w: (240, 240, 155) float32
t2f: (240, 240, 155) float32
seg: (240, 240, 155) float32


In [16]:
mri_3d = np.stack(
    [
        volumes["t1n"],
        volumes["t1c"],
        volumes["t2w"],
        volumes["t2f"]
    ],
    axis=0
)

print(
    "3D MRI shape:",
    mri_3d.shape
)

print(
    "3D MRI dtype:",
    mri_3d.dtype
)

print(
    "Mask shape:",
    seg.shape
)

print(
    "Mask values:",
    np.unique(seg)
)

print(
    "MRI memory:",
    round(
        mri_3d.nbytes / (1024 ** 2),
        2
    ),
    "MB"
)

print(
    "Mask memory:",
    round(
        seg.nbytes / (1024 ** 2),
        2
    ),
    "MB"
)

3D MRI shape: (4, 240, 240, 155)
3D MRI dtype: float32
Mask shape: (240, 240, 155)
Mask values: [0. 1. 2. 3.]
MRI memory: 136.23 MB
Mask memory: 34.06 MB


In [17]:
# ==================================================
# V4 3D Patch Test
# ==================================================

PATCH_D = 64
PATCH_H = 128
PATCH_W = 128

# Convert mask to binary
binary_mask = (
    seg > 0
).astype(np.float32)

# Take a central patch
z_start = (mri_3d.shape[1] - PATCH_D) // 2
y_start = (mri_3d.shape[2] - PATCH_H) // 2
x_start = (mri_3d.shape[3] - PATCH_W) // 2

image_patch = mri_3d[
    :,
    z_start:z_start + PATCH_D,
    y_start:y_start + PATCH_H,
    x_start:x_start + PATCH_W
]

mask_patch = binary_mask[
    z_start:z_start + PATCH_D,
    y_start:y_start + PATCH_H,
    x_start:x_start + PATCH_W
]

print("Full MRI:", mri_3d.shape)
print("Image patch:", image_patch.shape)
print("Mask patch:", mask_patch.shape)

print(
    "Patch MRI memory:",
    round(
        image_patch.nbytes / (1024 ** 2),
        2
    ),
    "MB"
)

print(
    "Patch mask memory:",
    round(
        mask_patch.nbytes / (1024 ** 2),
        2
    ),
    "MB"
)

print(
    "Patch mask values:",
    np.unique(mask_patch)
)

Full MRI: (4, 240, 240, 155)
Image patch: (4, 64, 128, 128)
Mask patch: (64, 128, 128)
Patch MRI memory: 16.0 MB
Patch mask memory: 4.0 MB
Patch mask values: [0. 1.]


In [18]:
import torch
import torch.nn as nn

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


In [19]:
import sys

V4_DIR = (
    PROJECT_ROOT
    / "src"
    / "v4_3d_unet_model"
)

sys.path.insert(
    0,
    str(V4_DIR)
)

from unet_3d import UNet3D

In [20]:
model_3d = UNet3D().to(device)

x = torch.from_numpy(
    image_patch
).unsqueeze(0).to(device)

y = torch.from_numpy(
    mask_patch
).unsqueeze(0).unsqueeze(1).to(device)

print("Input:", x.shape)
print("Target:", y.shape)

model_3d.train()

optimizer = torch.optim.AdamW(
    model_3d.parameters(),
    lr=1e-4
)

criterion = nn.BCEWithLogitsLoss()

optimizer.zero_grad()

logits = model_3d(x)

print(
    "Output:",
    logits.shape
)

loss = criterion(
    logits,
    y
)

print(
    "Loss:",
    loss.item()
)

loss.backward()

optimizer.step()

print("3D forward + backward successful")

Input: torch.Size([1, 4, 64, 128, 128])
Target: torch.Size([1, 1, 64, 128, 128])
Output: torch.Size([1, 1, 64, 128, 128])
Loss: 0.7274507284164429
3D forward + backward successful


In [22]:
from pathlib import Path
import glob
import numpy as np

PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v4"
)

In [26]:
from pathlib import Path
import numpy as np

TEST_DIR = Path(
    "/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test"
)

print("Directory exists:", TEST_DIR.exists())

files = sorted(
    TEST_DIR.glob("*.npz")
)

print("NPZ files found:", len(files))

if files:
    print("First file:", files[0])

Directory exists: True
NPZ files found: 43
First file: /Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test/BraTS-GLI-01250-000.npz


In [27]:
from pathlib import Path

old_patch = Path(
    "/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test/BraTS-GLI-01250-000.npz"
)

if old_patch.exists():
    old_patch.unlink()
    print("Removed old prototype patch")
else:
    print("Old prototype patch not found")

Removed old prototype patch


In [28]:
files = sorted(
    Path(
        "/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test"
    ).glob("*.npz")
)

print("Systematic patches:", len(files))

Systematic patches: 42


In [29]:
files = sorted(
    Path(
        "/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test"
    ).glob("*.npz")
)

tumor_pixel_counts = []

for path in files:
    data = np.load(path)
    pixels = int(data["mask"].sum())

    if pixels > 0:
        tumor_pixel_counts.append(pixels)

print("Total patches:", len(files))
print("Tumor patches:", len(tumor_pixel_counts))

print("Min tumor pixels:", min(tumor_pixel_counts))
print("Median tumor pixels:", np.median(tumor_pixel_counts))
print("Max tumor pixels:", max(tumor_pixel_counts))

print()
print("Tumor patches <100:", sum(x < 100 for x in tumor_pixel_counts))
print("Tumor patches 100–499:", sum(100 <= x < 500 for x in tumor_pixel_counts))
print("Tumor patches >=500:", sum(x >= 500 for x in tumor_pixel_counts))

Total patches: 42
Tumor patches: 18
Min tumor pixels: 3658
Median tumor pixels: 32484.0
Max tumor pixels: 65993

Tumor patches <100: 0
Tumor patches 100–499: 0
Tumor patches >=500: 18


In [30]:
# ==================================================
# Inspect V4 Patch Distribution
# ==================================================

PATCH_SIZE = (64, 128, 128)

SZ = 32
SY = 64
SX = 64

D, H, W = mri_3d.shape[1:]

z_starts = list(range(0, D - PATCH_SIZE[0] + 1, SZ))
y_starts = list(range(0, H - PATCH_SIZE[1] + 1, SY))
x_starts = list(range(0, W - PATCH_SIZE[2] + 1, SX))

if z_starts[-1] != D - PATCH_SIZE[0]:
    z_starts.append(D - PATCH_SIZE[0])

if y_starts[-1] != H - PATCH_SIZE[1]:
    y_starts.append(H - PATCH_SIZE[1])

if x_starts[-1] != W - PATCH_SIZE[2]:
    x_starts.append(W - PATCH_SIZE[2])

patch_info = []

for z in z_starts:
    for y in y_starts:
        for x in x_starts:

            patch_mask = binary_mask[
                z:z + PATCH_SIZE[0],
                y:y + PATCH_SIZE[1],
                x:x + PATCH_SIZE[2]
            ]

            tumor_voxels = int(
                patch_mask.sum()
            )

            if tumor_voxels > 0:

                patch_info.append({
                    "z": z,
                    "y": y,
                    "x": x,
                    "tumor_voxels": tumor_voxels
                })

print(
    "Tumor-containing patches:",
    len(patch_info)
)

for item in patch_info:
    print(item)

Tumor-containing patches: 18
{'z': 32, 'y': 0, 'x': 0, 'tumor_voxels': 3658}
{'z': 32, 'y': 0, 'x': 27, 'tumor_voxels': 3658}
{'z': 32, 'y': 64, 'x': 0, 'tumor_voxels': 32484}
{'z': 32, 'y': 64, 'x': 27, 'tumor_voxels': 32484}
{'z': 32, 'y': 112, 'x': 0, 'tumor_voxels': 32480}
{'z': 32, 'y': 112, 'x': 27, 'tumor_voxels': 32480}
{'z': 64, 'y': 0, 'x': 0, 'tumor_voxels': 9328}
{'z': 64, 'y': 0, 'x': 27, 'tumor_voxels': 9328}
{'z': 64, 'y': 64, 'x': 0, 'tumor_voxels': 65993}
{'z': 64, 'y': 64, 'x': 27, 'tumor_voxels': 65993}
{'z': 64, 'y': 112, 'x': 0, 'tumor_voxels': 65989}
{'z': 64, 'y': 112, 'x': 27, 'tumor_voxels': 65989}
{'z': 96, 'y': 0, 'x': 0, 'tumor_voxels': 5670}
{'z': 96, 'y': 0, 'x': 27, 'tumor_voxels': 5670}
{'z': 96, 'y': 64, 'x': 0, 'tumor_voxels': 33509}
{'z': 96, 'y': 64, 'x': 27, 'tumor_voxels': 33509}
{'z': 96, 'y': 112, 'x': 0, 'tumor_voxels': 33509}
{'z': 96, 'y': 112, 'x': 27, 'tumor_voxels': 33509}


In [31]:
print("Volume:", mri_3d.shape)

print("Z starts:", z_starts)
print("Y starts:", y_starts)
print("X starts:", x_starts)

print("Patch size:", PATCH_SIZE)

Volume: (4, 240, 240, 155)
Z starts: [0, 32, 64, 96, 128, 160, 176]
Y starts: [0, 64, 112]
X starts: [0, 27]
Patch size: (64, 128, 128)


In [32]:
mri_3d = np.stack(
    [
        volumes["t1n"],
        volumes["t1c"],
        volumes["t2w"],
        volumes["t2f"]
    ],
    axis=0
)

# Current: (C, H, W, Z)
# Convert: (C, Z, H, W)

mri_3d = np.transpose(
    mri_3d,
    (0, 3, 1, 2)
)

# Mask: (H, W, Z)
# Convert: (Z, H, W)

binary_mask = np.transpose(
    binary_mask,
    (2, 0, 1)
)

print("MRI:", mri_3d.shape)
print("Mask:", binary_mask.shape)

MRI: (4, 155, 240, 240)
Mask: (155, 240, 240)


In [33]:
from pathlib import Path

test_dir = Path(
    "/Users/abhra/Downloads/AI-ML/Resume/BraTS/data/processed/v4/test"
)

for file in test_dir.glob("*.npz"):
    file.unlink()

print("Old V4 test patches removed.")

Old V4 test patches removed.


In [36]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/Users/abhra/Downloads/AI-ML/Resume/BraTS"
)

V4_SRC = (
    PROJECT_ROOT
    / "src"
    / "v4_3d_unet_model"
)

sys.path.insert(
    0,
    str(V4_SRC)
)

from dataset import BraTS3DDataset

TEST_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v4"
    / "test"
)

dataset = BraTS3DDataset(
    TEST_DIR
)

print("Dataset size:", len(dataset))

image, mask = dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

print(
    "Mask values:",
    torch.unique(mask)
)

Found 36 3D patches
Dataset size: 36
Image shape: torch.Size([4, 64, 128, 128])
Mask shape: torch.Size([64, 128, 128])
Image dtype: torch.float32
Mask dtype: torch.float32
Mask values: tensor([0., 1.])


In [37]:
from torch.utils.data import DataLoader
import time

BATCH_SIZE = 1
NUM_WORKERS = 0

test_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

print("Dataset size:", len(dataset))
print("Batches:", len(test_loader))

start = time.time()

images, masks = next(
    iter(test_loader)
)

elapsed = time.time() - start

print()
print("Batch images:", images.shape)
print("Batch masks:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print(
    f"First batch loading time: {elapsed:.3f} sec"
)

Dataset size: 36
Batches: 36

Batch images: torch.Size([1, 4, 64, 128, 128])
Batch masks: torch.Size([1, 64, 128, 128])
Image dtype: torch.float32
Mask dtype: torch.float32
First batch loading time: 0.019 sec


In [38]:
print("Running 20 V4 DataLoader batches...")

start = time.time()

for i, (images, masks) in enumerate(test_loader):

    if i >= 20:
        break

elapsed = time.time() - start

batches = min(
    20,
    len(test_loader)
)

print()
print("Batches:", batches)
print(
    f"Total time: {elapsed:.2f} seconds"
)
print(
    f"Time/batch: {elapsed / batches:.3f} seconds"
)

Running 20 V4 DataLoader batches...

Batches: 20
Total time: 0.26 seconds
Time/batch: 0.013 seconds


In [39]:
import time
import torch
import torch.nn as nn

model_3d = UNet3D().to(device)

optimizer = torch.optim.AdamW(
    model_3d.parameters(),
    lr=1e-4
)

bce_loss = nn.BCEWithLogitsLoss()

model_3d.train()

print("Running 10 V4 training batches...")

start = time.time()

for i, (images, masks) in enumerate(test_loader):

    if i >= 10:
        break

    images = images.to(device)
    masks = masks.to(device)

    masks = masks.unsqueeze(1)

    optimizer.zero_grad()

    logits = model_3d(images)

    loss = bce_loss(
        logits,
        masks
    )

    loss.backward()

    optimizer.step()

elapsed = time.time() - start

batches = min(
    10,
    len(test_loader)
)

print()
print("Batches:", batches)
print(
    f"Total time: {elapsed:.2f} seconds"
)
print(
    f"Time/batch: {elapsed / batches:.2f} seconds"
)
print(
    f"Estimated 36-patch epoch: "
    f"{(elapsed / batches) * 36 / 60:.2f} minutes"
)

Running 10 V4 training batches...

Batches: 10
Total time: 20.08 seconds
Time/batch: 2.01 seconds
Estimated 36-patch epoch: 1.20 minutes


In [40]:
from pathlib import Path
import sys
import time
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(
    "/Users/abhra/Downloads/AI-ML/Resume/BraTS"
)

V4_SRC = (
    PROJECT_ROOT
    / "src"
    / "v4_3d_unet_model"
)

sys.path.insert(
    0,
    str(V4_SRC)
)

from dataset import BraTS3DDataset


# ==================================================
# Device
# ==================================================

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)


# ==================================================
# Dataset
# ==================================================

TRAIN_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v4"
    / "train"
)

VAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "v4"
    / "val"
)

train_dataset = BraTS3DDataset(
    TRAIN_DIR
)

val_dataset = BraTS3DDataset(
    VAL_DIR
)

print()
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


# ==================================================
# DataLoader
# ==================================================

BATCH_SIZE = 1
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print()
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# ==================================================
# Batch test
# ==================================================

images, masks = next(
    iter(train_loader)
)

print()
print("Images:", images.shape)
print("Masks:", masks.shape)
print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)
print("Mask values:", torch.unique(masks))


# ==================================================
# Loading benchmark
# ==================================================

print()
print("Running 100 V4 DataLoader batches...")

start = time.time()

for i, (images, masks) in enumerate(train_loader):

    if i >= 100:
        break

elapsed = time.time() - start

print()
print("Batches:", 100)
print(
    f"Total time: {elapsed:.2f} seconds"
)
print(
    f"Time/batch: {elapsed / 100:.3f} seconds"
)

print(
    f"Estimated full epoch loading time: "
    f"{elapsed / 100 * len(train_loader) / 60:.1f} minutes"
)

Device: mps
Found 35964 3D patches
Found 9072 3D patches

Training samples: 35964
Validation samples: 9072

Training batches: 35964
Validation batches: 9072

Images: torch.Size([1, 4, 64, 128, 128])
Masks: torch.Size([1, 64, 128, 128])
Image dtype: torch.float32
Mask dtype: torch.float32
Mask values: tensor([0.])

Running 100 V4 DataLoader batches...

Batches: 100
Total time: 1.30 seconds
Time/batch: 0.013 seconds
Estimated full epoch loading time: 7.8 minutes


In [41]:
from sampler import BalancedPatchSampler

sampler = BalancedPatchSampler(
    train_dataset,
    samples_per_epoch=4000,
    tumor_ratio=0.5
)

indices = sampler.sample_indices()

print()
print("Sampled patches:", len(indices))
print(
    "First 20 indices:",
    indices[:20]
)

Classifying V4 patches...
Tumor patches: 27051
Background patches: 8913

Sampled patches: 4000
First 20 indices: [35519, 28660, 14859, 6570, 35764, 28155, 8856, 16920, 25679, 13597, 22963, 19470, 33995, 33082, 14886, 22056, 1764, 5714, 21045, 8687]


In [48]:
import importlib
import sampler

print("Before reload:", sampler.BalancedPatchSampler.__init__.__code__.co_varnames)

importlib.reload(sampler)

print("After reload:", sampler.BalancedPatchSampler.__init__.__code__.co_varnames)

from sampler import BalancedPatchSampler

Before reload: ('self', 'dataset', 'samples_per_epoch', 'tumor_ratio', 'seed')
After reload: ('self', 'dataset', 'index_path', 'samples_per_epoch', 'tumor_ratio', 'seed')


In [49]:
import inspect

print(
    inspect.signature(
        BalancedPatchSampler
    )
)

(dataset, index_path, samples_per_epoch=4000, tumor_ratio=0.5, seed=42)


In [50]:
sampler = BalancedPatchSampler(
    dataset=train_dataset,
    index_path=TRAIN_INDEX,
    samples_per_epoch=4000,
    tumor_ratio=0.5,
    seed=42
)

Loaded patch index:
Tumor patches: 27051
Background patches: 8913


In [51]:
indices = sampler.sample_indices()

print("Sampled:", len(indices))
print("Unique indices:", len(set(indices)))
print("First 20:", indices[:20])

Sampled: 4000
Unique indices: 3726
First 20: [35519, 28660, 14859, 6570, 35764, 28155, 8856, 16920, 25679, 13597, 22963, 19470, 33995, 33082, 14886, 22056, 1764, 5714, 21045, 8687]
